# Instrument Master Quick View v0.1

Research-only notebook for quick visual inspection of `instrument_master_v0_1`.

Authority remains in the Data Foundation contracts, registry, validators and manifest.

In [1]:
from pathlib import Path
import json

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

root = Path(r"E:\TSIS\data\data_foundation_outputs\instrument_master")
parquet_path = root / "instrument_master_v0_1.parquet"
manifest_path = root / "_instrument_master_manifest_v0_1.json"
summary_path = root / "_instrument_master_summary_v0_1.csv"

assert parquet_path.exists(), parquet_path
assert manifest_path.exists(), manifest_path
assert summary_path.exists(), summary_path

In [2]:
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
summary = pd.read_csv(summary_path)

print("dataset_id:", manifest["dataset_id"])
print("schema_version:", manifest["schema_version"])
print("build_run_id:", manifest["build_run_id"])
print("output_sha256:", manifest["output_sha256"])

summary

dataset_id: instrument_master_v0_1
schema_version: instrument_master_v0_1
build_run_id: instrument_master_v0_1_20260621T145725Z
output_sha256: 69104387d2607306c3fa1740573d130db5e7c30b1d1527d3ee8a8d2b4d53c2d2


,metric,value
0,rows,4824
1,tickers,4824
2,duplicate_ticker_count,0
3,missing_instrument_id_count,0
4,invalid_window_count,0
5,non_common_stock_count,0
6,ticker_change_review_count,2721


In [11]:
df = pd.read_parquet(parquet_path)

print("rows:", len(df))
print("tickers:", df["ticker"].nunique())
print("columns:", len(df.columns))
print("schema_versions:", sorted(df["schema_version"].dropna().unique().tolist()))

identity_cols = [
    "instrument_id",
    "ticker",
    "identity_resolution_level",
    "ticker_identity_scope",
    "valid_from",
    "valid_to",
    "name",
    "primary_exchange",
    "ticker_type_code",
    "is_common_stock",
    "cik",
    "composite_figi",
    "share_class_figi",
    "lt1b_classification_1b",
    "ticker_change_event_count",
    "reference_snapshot_timing",
]

df[identity_cols].tail(3).T

rows: 4824
tickers: 4824
columns: 53
schema_versions: ['instrument_master_v0_1']


,4821,4822,4823
instrument_id,figi_share_class:BBG00YZ2VVK7,figi_share_class:BBG0077HPN83,figi_share_class:BBG001S7T7V0
ticker,ZWRK,ZY,ZYXI
identity_resolution_level,share_class_figi,share_class_figi,share_class_figi
ticker_identity_scope,lt1b_ticker_grain_v0_1,lt1b_ticker_grain_v0_1,lt1b_ticker_grain_v0_1
valid_from,2021-03-22 00:00:00,2021-04-22 00:00:00,2019-02-12 00:00:00
valid_to,2022-12-08 00:00:00,2022-10-19 00:00:00,2025-12-23 00:00:00
name,Z-Work Acquisition Corp. Class A Common Stock,Zymergen Inc. Common Stock,ZYNEX INC
primary_exchange,XNAS,XNAS,XNAS
ticker_type_code,CS,CS,CS
is_common_stock,True,True,True


In [4]:
pd.DataFrame(
    {
        "identity_resolution_level": df["identity_resolution_level"].value_counts(dropna=False),
    }
).join(
    pd.DataFrame({"pct": df["identity_resolution_level"].value_counts(normalize=True, dropna=False) * 100})
).round({"pct": 2})

,identity_resolution_level,pct
identity_resolution_level,,
share_class_figi,3556,73.71
cik_ticker,1265,26.22
composite_figi,3,0.06


In [5]:
df.loc[df["ticker_change_event_count"] > 0, identity_cols].head(20)

,instrument_id,ticker,identity_resolution_level,ticker_identity_scope,valid_from,valid_to,name,primary_exchange,ticker_type_code,is_common_stock,cik,composite_figi,share_class_figi,lt1b_classification_1b,ticker_change_event_count,reference_snapshot_timing
1,figi_share_class:BBG00ZKGK109,AAGR,share_class_figi,lt1b_ticker_grain_v0_1,2023-12-07,2024-09-25,African Agriculture Holdings Inc. Common Stock,XNAS,CS,True,0001848898,BBG00ZKGK0R2,BBG00ZKGK109,inactive_died_lt_1b,1,exact_anchor
4,figi_share_class:BBG001S5N8T1,AAME,share_class_figi,lt1b_ticker_grain_v0_1,2016-10-25,2026-03-09,Atlantic American Corp,XNAS,CS,True,0000008177,BBG000B9XB24,BBG001S5N8T1,active_lt_1b_last_classifiable,1,before_anchor
7,figi_share_class:BBG01223DLC1,AARD,share_class_figi,lt1b_ticker_grain_v0_1,2025-02-13,2026-03-09,"Aardvark Therapeutics, Inc. Common Stock",XNAS,CS,True,0001774857,BBG01223DLB2,BBG01223DLC1,active_lt_1b_last_classifiable,1,before_anchor
8,figi_share_class:BBG011MC2119,AATC,share_class_figi,lt1b_ticker_grain_v0_1,2021-07-21,2022-12-29,Autoscope Technologies Corporation Common Stock,XNAS,CS,True,0000943034,BBG011MC2100,BBG011MC2119,inactive_died_lt_1b,2,exact_anchor
10,figi_share_class:BBG004M1KJP3,ABAT,share_class_figi,lt1b_ticker_grain_v0_1,2023-09-21,2026-03-09,American Battery Technology Company Common Stock,XNAS,CS,True,0001576873,BBG004M1KJN5,BBG004M1KJP3,active_lt_1b_last_classifiable,2,before_anchor
11,figi_share_class:BBG001S8T7K0,ABEO,share_class_figi,lt1b_ticker_grain_v0_1,2016-10-25,2026-03-09,Abeona Therapeutics Inc. Common Stock,XNAS,CS,True,0000318306,BBG000DT5D52,BBG001S8T7K0,active_lt_1b_last_classifiable,1,before_anchor
15,cik_ticker:0001957489:ABLV,ABLV,cik_ticker,lt1b_ticker_grain_v0_1,2023-08-21,2026-03-09,Able View Global Inc. Class B Ordinary Shares,XNAS,CS,True,0001957489,NaN,NaN,active_lt_1b_last_classifiable,1,exact_anchor
16,figi_share_class:BBG0058YJ326,ABOS,share_class_figi,lt1b_ticker_grain_v0_1,2021-07-01,2026-03-09,"Acumen Pharmaceuticals, Inc. Common Stock",XNAS,CS,True,0001576885,BBG0058YJ317,BBG0058YJ326,active_lt_1b_last_classifiable,1,before_anchor
18,figi_share_class:BBG011N54DC3,ABSI,share_class_figi,lt1b_ticker_grain_v0_1,2021-07-22,2026-03-09,Absci Corporation Common Stock,XNAS,CS,True,0001672688,BBG011N54CJ8,BBG011N54DC3,active_lt_1b_last_classifiable,1,before_anchor
19,figi_share_class:BBG00MS8GZQ0,ABTC,share_class_figi,lt1b_ticker_grain_v0_1,2025-09-03,2026-03-09,American Bitcoin Corp. Class A Common Stock,XNAS,CS,True,0001755953,BBG00MS8GZP1,BBG00MS8GZQ0,active_lt_1b_last_classifiable,1,before_anchor
